In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import torch
import matplotlib.pyplot as plt

PROJECT_DIR = '/content/drive/MyDrive/gpr_fwi_diffusion'
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())
print("Files in Drive:", os.listdir(PROJECT_DIR))

Mounted at /content/drive
Working directory: /content/drive/MyDrive/gpr_fwi_diffusion
Files in Drive: ['groundtruth.png', 'observed_data.png', 'obs_data.npy', 'trace_cols.npy', 'gpr_synthetic_dataset.pt', 'loss_history.pt', 'diffusion_unet_weights.pt', 'loss_curves.npy', 'all_metrics.npy', 'inversion_results.npy', 'obs_data_hard.npy', 'trace_cols_hard.npy', 'inversion_results_hard.npy', 'all_metrics_hard.npy', 'figure5_comparison_preview.png', 'obs_data_v2.npy', 'trace_cols_v2.npy', 'all_metrics_v2.npy', 'inversion_results_v2.npy', 'obs_data_v3.npy', 'trace_cols_v3.npy', 'eps_true.npy', 'sig_true.npy', 'figure5_comparison_v3.png', 'inversion_results_v3.npy', 'figure5_comparison_400iter.png', 'all_metrics_v3.npy', 'figure5_final.png']


In [2]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# ---- Normalization bounds (locked throughout the paper) ----
EPS_MIN, EPS_MAX = 2.0, 25.0
SIG_MIN, SIG_MAX = 0.0001, 0.05

def normalize(x, lo, hi):
    return 2 * (np.clip(x, lo, hi) - lo) / (hi - lo) - 1

def denormalize(x, lo, hi):
    return (x + 1) / 2 * (hi - lo) + lo

# ---- U-Net architecture (identical to Figure 4/5) ----
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = t[:, None].float() * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_ch)
        self.block1 = nn.Sequential(
            nn.GroupNorm(8, in_ch), nn.SiLU(),
            nn.Conv2d(in_ch, out_ch, 3, padding=1))
        self.block2 = nn.Sequential(
            nn.GroupNorm(8, out_ch), nn.SiLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1))
        self.res_conv = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t_emb):
        h = self.block1(x)
        h = h + self.time_mlp(t_emb)[:, :, None, None]
        h = self.block2(h)
        return h + self.res_conv(x)

class SimpleUNet(nn.Module):
    def __init__(self, in_ch=2, base_ch=64, time_emb_dim=128):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim), nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim))
        self.in_conv = nn.Conv2d(in_ch, base_ch, 3, padding=1)
        self.down1 = ResidualBlock(base_ch, base_ch, time_emb_dim)
        self.pool1 = nn.Conv2d(base_ch, base_ch, 4, stride=2, padding=1)
        self.down2 = ResidualBlock(base_ch, base_ch*2, time_emb_dim)
        self.pool2 = nn.Conv2d(base_ch*2, base_ch*2, 4, stride=2, padding=1)
        self.down3 = ResidualBlock(base_ch*2, base_ch*4, time_emb_dim)
        self.pool3 = nn.Conv2d(base_ch*4, base_ch*4, 4, stride=2, padding=1)
        self.bottleneck = ResidualBlock(base_ch*4, base_ch*4, time_emb_dim)
        self.up3 = nn.ConvTranspose2d(base_ch*4, base_ch*4, 4, stride=2, padding=1)
        self.dec3 = ResidualBlock(base_ch*8, base_ch*2, time_emb_dim)
        self.up2 = nn.ConvTranspose2d(base_ch*2, base_ch*2, 4, stride=2, padding=1)
        self.dec2 = ResidualBlock(base_ch*4, base_ch, time_emb_dim)
        self.up1 = nn.ConvTranspose2d(base_ch, base_ch, 4, stride=2, padding=1)
        self.dec1 = ResidualBlock(base_ch*2, base_ch, time_emb_dim)
        self.out_conv = nn.Sequential(
            nn.GroupNorm(8, base_ch), nn.SiLU(),
            nn.Conv2d(base_ch, in_ch, 3, padding=1))
    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        x0 = self.in_conv(x)
        d1 = self.down1(x0, t_emb)
        d2 = self.down2(self.pool1(d1), t_emb)
        d3 = self.down3(self.pool2(d2), t_emb)
        b  = self.bottleneck(self.pool3(d3), t_emb)
        u3 = self.dec3(torch.cat([self.up3(b),  d3], dim=1), t_emb)
        u2 = self.dec2(torch.cat([self.up2(u3), d2], dim=1), t_emb)
        u1 = self.dec1(torch.cat([self.up1(u2), d1], dim=1), t_emb)
        return self.out_conv(u1)

class DiffusionSchedule:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02, device='cuda'):
        self.timesteps = timesteps
        self.betas = torch.linspace(beta_start, beta_end, timesteps, device=device)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_ac   = self.sqrt_alphas_cumprod[t][:, None, None, None]
        sqrt_omac = self.sqrt_one_minus_alphas_cumprod[t][:, None, None, None]
        return sqrt_ac * x0 + sqrt_omac * noise

# ---- Load trained model ----
model    = SimpleUNet().to(device)
schedule = DiffusionSchedule(timesteps=1000, device=device)
model.load_state_dict(torch.load('diffusion_unet_weights.pt', map_location=device))
model.eval()
print("Diffusion model loaded successfully.")

# ---- Forward model (Born approximation) ----
def forward_model(eps, sig, n_traces=32, noise_level=0.0):
    H, W = eps.shape
    t = np.linspace(-1, 1, 21)
    f0 = 0.3
    wavelet = (1 - 2*(np.pi*f0*t)**2) * np.exp(-(np.pi*f0*t)**2)
    trace_cols = np.linspace(0, W-1, n_traces, dtype=int)
    data = np.zeros((n_traces, H), dtype=np.float32)
    for i, col in enumerate(trace_cols):
        impedance   = 1.0 / np.sqrt(eps[:, col])
        refl        = np.diff(impedance, prepend=impedance[0])
        attenuation = np.exp(-np.cumsum(sig[:, col]) * 2.0)
        refl        = refl * attenuation
        data[i]     = np.convolve(refl, wavelet, mode='same')
    if noise_level > 0:
        data += noise_level * np.std(data) * np.random.randn(*data.shape)
    return data.astype(np.float32), trace_cols

def compute_gradient(eps, sig, obs_data, trace_cols):
    H, W = eps.shape
    t = np.linspace(-1, 1, 21)
    f0 = 0.3
    wavelet   = (1 - 2*(np.pi*f0*t)**2) * np.exp(-(np.pi*f0*t)**2)
    pred_data = forward_model(eps, sig, n_traces=len(trace_cols),
                               noise_level=0.0)[0]
    residual  = pred_data - obs_data
    misfit    = 0.5 * np.sum(residual**2)
    grad_eps  = np.zeros_like(eps)
    grad_sig  = np.zeros_like(sig)
    for i, col in enumerate(trace_cols):
        adj         = np.convolve(residual[i], wavelet[::-1], mode='same')
        attenuation = np.exp(-np.cumsum(sig[:, col]) * 2.0)
        d_imp_d_eps = -0.5 / (eps[:, col]**1.5)
        for d in range(H):
            grad_eps[d, col] += adj[d] * attenuation[d] * d_imp_d_eps[d]
        for d in range(H):
            grad_sig[d, col] += -2.0 * adj[d] * attenuation[d] * \
                                  np.sum(adj[:d+1])
    return grad_eps, grad_sig, misfit

@torch.no_grad()
def diffusion_project_gentle(eps_arr, sig_arr, t_start=50):
    eps_n = normalize(eps_arr, EPS_MIN, EPS_MAX)
    sig_n = normalize(sig_arr, SIG_MIN, SIG_MAX)
    x0    = torch.from_numpy(
        np.stack([eps_n, sig_n], axis=0)[None]).float().to(device)
    t_tensor = torch.tensor([t_start], device=device)
    x        = schedule.q_sample(x0, t_tensor, torch.randn_like(x0))
    for t_cur in reversed(range(0, t_start)):
        t_cur_tensor = torch.tensor([t_cur], device=device)
        beta_t    = schedule.betas[t_cur]
        alpha_t   = schedule.alphas[t_cur]
        alpha_bar = schedule.alphas_cumprod[t_cur]
        pred_noise = model(x, t_cur_tensor)
        if t_cur > 0:
            coef = beta_t / torch.sqrt(1 - alpha_bar)
            x    = (1 / torch.sqrt(alpha_t)) * (x - coef * pred_noise)
        else:
            x = (x - torch.sqrt(1 - alpha_bar) * pred_noise) / \
                 torch.sqrt(alpha_bar)
            x = torch.clamp(x, -1, 1)
    x_d     = x.squeeze(0).cpu().numpy()
    eps_den = np.clip(denormalize(x_d[0], EPS_MIN, EPS_MAX), EPS_MIN, EPS_MAX)
    sig_den = np.clip(denormalize(x_d[1], SIG_MIN, SIG_MAX), SIG_MIN, SIG_MAX)
    return eps_den, sig_den

from skimage.metrics import structural_similarity as skssim
def compute_metrics(recon, true):
    rmse = np.sqrt(np.mean((recon - true)**2))
    ssim_val = skssim(true, recon, data_range=true.max() - true.min())
    return rmse, ssim_val

print("All definitions ready.")

Device: cpu
Diffusion model loaded successfully.
All definitions ready.
